# REM_Turku DL baseline, handoff notebook

**Paper:** Paul Barbaste, Riemannian geometry EEG classification, NeurIPS 2026.

`evaluate()` is separable from the model. Pass any `fit_predict` callable and you get the
same folds, seeds and per-subject table the DL baseline used. Matched by construction.

Runs locally on a 6-subject extract (87 MB, CPU, minutes). Full corpus on the cluster at
`/orcd/scratch/orcd/010/ninon/reaDream/turku/`.

## 1. Dataset

REM_Turku (Sikka, Revonsuo, Noreika, Valli), figshare 10.6084/m9.figshare.23274596.v2.
24 EEG channels at 10/10, 500 Hz, 2 min of pre-awakening REM per file, serial awakenings.
133 EDFs, 17 usable subjects, 121 awakenings with labels.

Preprocessing: `prepare_remturku.py`, version `remturku-v1-sikka2019`, reproducing
Sikka et al. 2019 (J Neurosci 39:4775): FIR 0.5 to 45 Hz, average reference, ICA with
EOG-driven rejection, 2 s epochs at 50% overlap, epochs above 200 uV dropped, CSD and
pre-CSD both saved. Validated against the paper: anger in 40% of dreams vs 41% published,
interest 88% vs 88%.

Labels: mDES self-report, 20 items rated 0 to 4 by the sleeper at each awakening, mapped
to Hall and Van de Castle by presence (`SR > 0`).

In [ ]:
import json
import numpy as np
import torch
import torch.nn as nn
from pathlib import Path
from torch.utils.data import TensorDataset, DataLoader

DATA = Path("data")
meta = json.loads((DATA / "remturku_local_meta.json").read_text())
npz  = np.load(DATA / "remturku_local_extract.npz")
print(f"{len(meta)} awakenings, {len({m['subject'] for m in meta})} subjects (local extract)")
for t in ("anger", "apprehension", "confusion"):
    p = np.mean([m[t] for m in meta])
    print(f"  {t:<14} rate={p:.3f}  chance_uniform=0.500  chance_majority={max(p,1-p):.3f}")

## 2. Pre-declared configuration

Set before the first run. Changing it after seeing results invalidates the statistics.

Family of 3, one per dataset, Riemannian against the better DL arm. alpha = 0.0167.

Interpretive floor, separate from significance: **3 accuracy points**. Tuning ShallowConv
on this family moved it from +3.43 to +6.66 in-house, so a smaller margin sits inside the
tuning gap. A p below 0.0167 with a 1.5-point margin is not a claim.

In [ ]:
SEEDS = [0, 1, 2, 3, 4]
COMPARISON_FAMILY = 3
ALPHA = 0.05 / COMPARISON_FAMILY          # 0.0167
MIN_CREDIBLE_MARGIN = 0.03                # 3 accuracy points, the in-house tuning gap
SEED_CONTROLS_FOLDS = True                # seed drives init AND fold assignment

# Detectable-effect floors, computed on the label file with no EEG involved
FLOORS = {"anger": 0.65, "apprehension": 0.62, "confusion": 0.65}
print(f"alpha={ALPHA:.4f}  min credible margin={MIN_CREDIBLE_MARGIN}  floors={FLOORS}")

## 3. The evaluation harness

`evaluate(fit_predict, target)` takes any callable with signature
`fit_predict(Xtr, ytr, Xte, seed) -> predictions` and returns the per-subject table.

DL baseline passes a ShallowConv factory. Paul passes a pyRiemann pipeline. Same subjects,
folds, seeds, scaling and labels.

In [ ]:
def load_target(target, kind="raw"):
    """Returns per-awakening arrays, labels, subjects. One row per EPOCH downstream."""
    X, y, s = [], [], []
    for m in meta:
        X.append(npz[m["filename"]]); y.append(m[target]); s.append(m["subject"])
    return X, np.array(y), np.array(s)


def to_windows(X, y, s):
    xs, ys, ss = [], [], []
    for a, lab, sub in zip(X, y, s):
        for e in a:
            xs.append(e); ys.append(lab); ss.append(sub)
    return np.stack(xs)[:, :, :, None], np.array(ys), np.array(ss)


def standardize(Xtr, Xte):
    """Volts -> microvolts, then per-channel z-score on TRAINING statistics only.

    Not cosmetic. MNE returns EDF in volts (std ~9e-06 here). A CNN with Glorot init on
    1e-5 input has near-zero gradients, collapses to majority class and returns exactly
    0.500 balanced accuracy. That is what the first launch of this experiment produced.
    Training-fold statistics only: whole-set statistics leak the held-out subject's
    amplitude distribution into the scaler.
    """
    Xtr, Xte = Xtr * 1e6, Xte * 1e6
    mu = Xtr.mean(axis=(0, 2), keepdims=True)
    sd = Xtr.std(axis=(0, 2), keepdims=True) + 1e-8
    return (Xtr - mu) / sd, (Xte - mu) / sd


def bal_acc(pred, true):
    tp = ((pred == 1) & (true == 1)).sum(); fn = ((pred == 0) & (true == 1)).sum()
    tn = ((pred == 0) & (true == 0)).sum(); fp = ((pred == 1) & (true == 0)).sum()
    se = tp / (tp + fn) if tp + fn else 0.0
    sp = tn / (tn + fp) if tn + fp else 0.0
    return float((se + sp) / 2)


def evaluate(fit_predict, target, seeds=SEEDS, verbose=True):
    """Leave-one-subject-out. THE harness. Pass any fit_predict(Xtr,ytr,Xte,seed)->preds."""
    X, y, s = load_target(target)
    subs = sorted(set(s))
    per_subject, folds = {}, {}
    for seed in seeds:
        for held in subs:
            tr, te = s != held, s == held
            if y[te].sum() in (0, te.sum()):
                continue                       # single-class test fold, undefined
            Xtr, ytr, _ = to_windows([X[i] for i in np.where(tr)[0]], y[tr], s[tr])
            Xte, yte, _ = to_windows([X[i] for i in np.where(te)[0]], y[te], s[te])
            Xtr, Xte = standardize(Xtr, Xte)
            rng = np.random.default_rng(seed)
            perm = rng.permutation(len(ytr))   # seed drives folds too
            Xtr, ytr = Xtr[perm], ytr[perm]
            folds.setdefault(seed, {})[held] = {"n_train": int(len(ytr)), "n_test": int(len(yte))}
            pred = fit_predict(Xtr, ytr, Xte, seed)
            a = bal_acc(np.asarray(pred), yte)
            per_subject.setdefault(held, {})[seed] = a
            if verbose: print(f"  seed={seed} held={held} n_test={len(yte)} bal.acc={a:.3f}")
    return per_subject, folds


def summarize(per_subject):
    """Average across seeds WITHIN subject first, then between-subject SD.
    Conflating seed variance with subject variance is the error reviewers catch."""
    subs = sorted(per_subject)
    means = np.array([np.mean(list(per_subject[s].values())) for s in subs])
    mc = np.array([np.std(list(per_subject[s].values()), ddof=1)
                   if len(per_subject[s]) > 1 else 0.0 for s in subs])
    return {"subjects": subs, "per_subject_mean": means.tolist(),
            "mean": float(means.mean()), "between_subject_sd": float(means.std(ddof=1)),
            "mc_sd_across_seeds_median": float(np.median(mc)), "n": len(subs)}

## 4. DL baseline: ShallowConvNet

Architecture imported unmodified. Its chain is `conv_time -> conv_spat -> SQUARE ->
AvgPool -> LOG`, which is log band power, the operation EEGNet cannot express.

Training regime copied verbatim from `tuning_p10_v3.py` (`tuning_core.py` on the cluster):
OneCycleLR stepped per batch, patience 25, `min_delta`, best-state restore, grad clip 1.0.
Loss is `NLLLoss`, not p10_v3's `CrossEntropyLoss`: the model ends in `LogSoftmax` and
CrossEntropy would double-count the softmax. `tuning_p12_stable.py` is the corrected variant.

In [ ]:
import sys; sys.path.insert(0, "../riemann")
from ShallowConv_Embedding_version import ShallowConv_Embedding

def make_shallow_fit_predict(cfg):
    def fit_predict(Xtr, ytr, Xte, seed):
        torch.manual_seed(seed); np.random.seed(seed)
        n_ch, n_t = Xtr.shape[1], Xtr.shape[2]
        model = ShallowConv_Embedding(in_chans=n_ch, n_classes=2,
                                      input_window_samples=n_t, drop_prob=cfg["drop_prob"])
        loss_fn = nn.NLLLoss()
        opt = torch.optim.AdamW(model.parameters(), lr=cfg["lr"],
                                weight_decay=cfg["weight_decay"])
        k = max(1, int(0.2 * len(ytr)))
        dl = lambda X, y, sh: DataLoader(TensorDataset(torch.tensor(X),
                              torch.tensor(y, dtype=torch.long)),
                              batch_size=cfg["batch_size"], shuffle=sh)
        tr, va = dl(Xtr[k:], ytr[k:], True), dl(Xtr[:k], ytr[:k], False)
        sch = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=cfg["lr"]*10,
                epochs=cfg["epochs"], steps_per_epoch=max(1,len(tr)),
                pct_start=cfg["pct_start"], three_phase=cfg["three_phase"])
        best, best_state, bad = -1.0, None, 0
        for _ in range(cfg["epochs"]):
            model.train()
            for xb, yb in tr:
                opt.zero_grad(); loss = loss_fn(model(xb.float()), yb); loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                opt.step(); sch.step()
            model.eval(); c = n = 0
            with torch.no_grad():
                for xb, yb in va:
                    c += (model(xb.float()).argmax(1) == yb).sum().item(); n += len(yb)
            acc = c / max(n, 1)
            if acc > best + cfg["min_delta"]:
                best, bad = acc, 0
                best_state = {k_: v.clone() for k_, v in model.state_dict().items()}
            else:
                bad += 1
                if bad >= cfg["patience"]: break
        if best_state: model.load_state_dict(best_state)
        model.eval()
        with torch.no_grad():
            return model(torch.tensor(Xte).float()).argmax(1).numpy()
    return fit_predict

CFG = dict(drop_prob=0.6, lr=3e-4, weight_decay=1e-3, batch_size=64,
           epochs=30, patience=10, min_delta=0.001, pct_start=0.3, three_phase=True)
print("CFG (local quick-run; the cluster run sweeps 26 dimensions):", CFG)

## 5. Run

On the local extract with `epochs=30` this is a few minutes on CPU. Change `target` to
`anger` or `confusion` for the contrast.

In [ ]:
per_subject, folds = evaluate(make_shallow_fit_predict(CFG), target="apprehension",
                              seeds=[0], verbose=True)
s = summarize(per_subject)
print(f"\nbal.acc mean={s['mean']:.3f}  between-subject sd={s['between_subject_sd']:.3f}  n={s['n']}")
print(f"floor for apprehension = {FLOORS['apprehension']}  chance = 0.500")

## 6. Paul's entry point

Replace the factory, keep everything else.

```python
from pyriemann.estimation import Covariances
from pyriemann.tangentspace import TangentSpace
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression

def riemann_fit_predict(Xtr, ytr, Xte, seed):
    Xtr2, Xte2 = Xtr[..., 0], Xte[..., 0]          # (N, ch, time)
    pipe = make_pipeline(Covariances(estimator="oas"),
                         TangentSpace(metric="riemann"),
                         LogisticRegression(max_iter=1000, random_state=seed))
    pipe.fit(Xtr2, ytr)
    return pipe.predict(Xte2)

per_subject_r, _ = evaluate(riemann_fit_predict, target="apprehension")
```

The Riemannian arm has hyperparameters too: shrinkage, filter bank, metric, tangent
reference. Either sweep them on the same dev split with the same protocol, or fix all of
them at pyRiemann defaults and name each value. Do not mix.

## 7. Cluster results

Full corpus, 17 subjects. Sweep over 26 hyperparameter dimensions on 6 dev subjects,
config frozen, LOSO on the remaining 11.

| target | model | LOSO bal.acc | chance | floor | status |
|---|---|---|---|---|---|
| anger | ShallowConv | 0.479 | 0.500 | 0.65 | below chance |
| anger | EEGNet | 0.460 | 0.500 | 0.65 | below chance |
| **apprehension** | **ShallowConv** | **0.665** | 0.500 | 0.62 | **clears floor** |
| apprehension | EEGNet | 0.608 | 0.500 | 0.62 | just under |
| confusion | both | running | 0.500 | 0.65 | |

apprehension/ShallowConv per subject: `[0.40, 0.44, 0.61, 0.69, 0.75, 0.76, 0.77, 0.77, 0.79]`.

Not a result yet, for three reasons. The apprehension cells were incomplete when read
(26 and 45 of 55 LOSO rows, 9 of 11 subjects). No permutation null has returned; 5 are
running. And test exceeded dev (0.665 vs 0.568), which needs explaining before it is
trusted; the plausible account is that dev trains on 5 subjects against 10 in the outer loop.

Anger below chance on both architectures is informative: it is what no signal looks like,
which makes the apprehension contrast harder to dismiss.